<a href="https://colab.research.google.com/github/JaimeTS/hyrox-analysis/blob/main/notebooks/01_scraping_hyrox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping de resultados HYROX

Objetivo: descargar los resultados de una prueba concreta desde HyResult y convertirlos en una base de datos estructurada para análisis de rendimiento, pacing y diferencias por edad y sexo.

## 1. Carga de librerías

Cargamos las librerías necesarias para descargar páginas web, leer HTML, estructurar datos en tablas y controlar pausas entre peticiones.

In [70]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import math
import time

## 2. Parámetros iniciales

Definimos la URL base de HyResult, la URL del evento y la URL de la categoría que vamos a descargar inicialmente.

In [71]:
BASE_URL = "https://www.hyresult.com"

event_url = "https://www.hyresult.com/event/s8-2026-barcelona"
ranking_url = "https://www.hyresult.com/ranking/s8-2026-barcelona-hyrox-men"

headers = {
    "User-Agent": "Mozilla/5.0"
}

## 3. Extracción de enlaces a rankings del evento

La página del evento contiene enlaces a las distintas divisiones de la prueba. Extraemos esos enlaces para poder automatizar posteriormente la descarga de varias categorías.

In [72]:
response_event = requests.get(event_url, headers=headers)
soup_event = BeautifulSoup(response_event.text, "html.parser")

links = []

for a in soup_event.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(strip=True)
    links.append((texto, href))

links_df = pd.DataFrame(links, columns=["texto", "href"])

ranking_links = links_df[
    links_df["href"].str.contains("/ranking/", case=False, na=False)
].drop_duplicates()

ranking_links

,texto,href
40,Ranking,/ranking/s8-2026-barcelona-hyrox-men
41,Ranking,/ranking/s8-2026-barcelona-hyrox-women
42,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-men
43,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-women
44,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-mixed
45,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-men
46,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-women
47,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-doubles-men
48,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-doubles-w...
49,Ranking,/ranking/s8-2026-barcelona-hyrox-team-relay-men


## 4. Función robusta para extraer una página del ranking

HyResult presenta normalmente un atleta por fila, aunque algunas posiciones aparecen divididas en varias filas consecutivas. Esta función reconstruye automáticamente esos casos para obtener todos los participantes de la página.

In [75]:
def extraer_ranking_pagina(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.find_all("tr")
    resultados = []

    i = 0

    while i < len(rows):
        cells = rows[i].find_all("td")
        textos = [c.get_text(" ", strip=True) for c in cells]

        # ==========================================================
        # CASO NORMAL
        # ==========================================================
        if len(cells) >= 7:

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            name_cell = cells[3]
            link_athlete = name_cell.find("a")
            img_flag = name_cell.find("img")

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = cells[4].get_text(strip=True)
            tiempo = cells[5].get_text(strip=True)

            analyze_link = cells[6].find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

            i += 1
            continue

        # ==========================================================
        # CASO ESPECIAL A
        # ==========================================================
        if len(cells) >= 3 and i + 4 < len(rows):

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            row_nombre = rows[i + 1]
            row_grupo = rows[i + 2]
            row_tiempo = rows[i + 3]
            row_analyze = rows[i + 4]

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 5
                continue

        # ==========================================================
        # CASO ESPECIAL C
        # (fila vacía + datos repartidos)
        # ==========================================================
        if len(cells) == 1 and textos == [""] and i + 6 < len(rows):

            row_pos = rows[i + 1]
            row_pos_ag = rows[i + 2]
            row_nombre = rows[i + 3]
            row_grupo = rows[i + 4]
            row_tiempo = rows[i + 5]
            row_analyze = rows[i + 6]

            posicion = row_pos.get_text(" ", strip=True)
            posicion_ag = row_pos_ag.get_text(" ", strip=True)
            nombre = row_nombre.get_text(" ", strip=True)
            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if (
                posicion.isdigit()
                and posicion_ag.isdigit()
                and nombre
                and tiempo
                and analyze_url
            ):

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": None,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": None,
                    "analyze_url": analyze_url
                })

                i += 7
                continue

        # ==========================================================
        # CASO ESPECIAL B
        # ==========================================================
        if len(cells) >= 2 and i + 5 < len(rows):

            posicion = cells[1].get_text(strip=True)

            row_pos_ag = rows[i + 1]
            row_nombre = rows[i + 2]
            row_grupo = rows[i + 3]
            row_tiempo = rows[i + 4]
            row_analyze = rows[i + 5]

            posicion_ag = row_pos_ag.get_text(" ", strip=True)

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 6
                continue

        i += 1

    df = pd.DataFrame(resultados)

    if len(df) > 0:

        df["athlete_url"] = df["athlete_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

        df["analyze_url"] = df["analyze_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

    return df

## 5. Prueba de extracción de una página

Probamos la función con la primera página y comprobamos que recupera las 100 posiciones.

In [76]:
df_pagina_1 = extraer_ranking_pagina(ranking_url)

print("Filas extraídas:", len(df_pagina_1))

df_pagina_1.head(10)

Filas extraídas: 100


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
0,1,1,Gaetan Gianni,France,16-24,53:44,https://www.hyresult.com/athlete/gaetan-gianni,https://www.hyresult.com/result/LR3MS4JI502C48
1,2,1,Emilio Aguayo,Spain,35-39,56:38,https://www.hyresult.com/athlete/emilio-aguayo,https://www.hyresult.com/result/LR3MS4JI501875
2,3,1,Loic Cebelieu,France,25-29,57:17,https://www.hyresult.com/athlete/loic-cebelieu,https://www.hyresult.com/result/LR3MS4JI5010E6
3,4,1,Ferran Bochaca Sabarich,Spain,30-34,57:24,https://www.hyresult.com/athlete/ferran-bochac...,https://www.hyresult.com/result/LR3MS4JI501D84
4,5,2,Alberto Casado Aroca,Spain,30-34,57:38,https://www.hyresult.com/athlete/alberto-casad...,https://www.hyresult.com/result/LR3MS4JI502972
5,6,3,Sebastien Rajkowski,France,30-34,58:10,https://www.hyresult.com/athlete/sebastien-raj...,https://www.hyresult.com/result/LR3MS4JI5022B0
6,7,4,Mehdi Berrouigat,France,30-34,58:19,https://www.hyresult.com/athlete/mehdi-berrouigat,https://www.hyresult.com/result/LR3MS4JI503639
7,8,2,Pau Nacenta Merinas,Spain,25-29,58:35,https://www.hyresult.com/athlete/pau-nacenta-m...,https://www.hyresult.com/result/LR3MS4JI50188C
8,9,3,Aaron Timmerman,Belgium,25-29,58:38,https://www.hyresult.com/athlete/aaron-timmerman,https://www.hyresult.com/result/LR3MS4JI500DE4
9,10,4,Walter Paladina,Italy,25-29,58:41,https://www.hyresult.com/athlete/walter-paladina,https://www.hyresult.com/result/LR3MS4JI502477


## 6. Función para descargar una categoría completa

HyResult pagina los rankings en bloques de 100 participantes. Esta función recorre todas las páginas de una categoría y concatena los resultados en un único DataFrame.

In [77]:
def descargar_categoria_completa(ranking_url, total_atletas, pausa=0.5):
    participantes_por_pagina = 100
    num_paginas = math.ceil(total_atletas / participantes_por_pagina)

    dfs = []

    for pagina in range(1, num_paginas + 1):
        url = f"{ranking_url}?p={pagina}"

        print(f"Descargando página {pagina}/{num_paginas}")

        df_pagina = extraer_ranking_pagina(url)
        dfs.append(df_pagina)

        time.sleep(pausa)

    df_categoria = pd.concat(dfs, ignore_index=True)

    return df_categoria

## 7. Descarga de HYROX Men Barcelona 2026

Aplicamos la función a la categoría HYROX Men del evento Barcelona 2026. En la página del evento esta categoría aparece con 2350 atletas.

In [78]:
total_atletas = 2350

df_categoria = descargar_categoria_completa(
    ranking_url=ranking_url,
    total_atletas=total_atletas,
    pausa=0.5
)

print("Participantes extraídos:", len(df_categoria))

df_categoria.tail()

Descargando página 1/24
Descargando página 2/24
Descargando página 3/24
Descargando página 4/24
Descargando página 5/24
Descargando página 6/24
Descargando página 7/24
Descargando página 8/24
Descargando página 9/24
Descargando página 10/24
Descargando página 11/24
Descargando página 12/24
Descargando página 13/24
Descargando página 14/24
Descargando página 15/24
Descargando página 16/24
Descargando página 17/24
Descargando página 18/24
Descargando página 19/24
Descargando página 20/24
Descargando página 21/24
Descargando página 22/24
Descargando página 23/24
Descargando página 24/24
Participantes extraídos: 2349


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
2344,2346,23,Carlos Barreto,Puerto Rico,60-64,2:36:53,https://www.hyresult.com/athlete/carlos-barreto,https://www.hyresult.com/result/LR3MS4JI5036F1
2345,2347,24,Emmanuel Weill,France,60-64,2:38:29,https://www.hyresult.com/athlete/emmanuel-weill,https://www.hyresult.com/result/LR3MS4JI503BBC
2346,2348,286,Mo Mo,United States,40-44,2:51:02,https://www.hyresult.com/athlete/mo-mo,https://www.hyresult.com/result/LR3MS4JI503D47
2347,2349,159,Nicolas Taillard,France,45-49,2:53:32,https://www.hyresult.com/athlete/nicolas-taillard,https://www.hyresult.com/result/LR3MS4JI501C3E
2348,2350,461,Avisek Mohanty,India,35-39,3:08:24,https://www.hyresult.com/athlete/avisek-mohanty,https://www.hyresult.com/result/LR3MS4JI50381A


## 8. Control de calidad de la extracción

Comprobamos si el número de participantes extraídos coincide con el total esperado y si hay posiciones duplicadas o faltantes.

In [79]:
df_categoria["posicion"] = pd.to_numeric(df_categoria["posicion"], errors="coerce")
df_categoria["posicion_grupo_edad"] = pd.to_numeric(df_categoria["posicion_grupo_edad"], errors="coerce")

print("Total esperado:", total_atletas)
print("Total extraído:", len(df_categoria))

print("Posición mínima:", df_categoria["posicion"].min())
print("Posición máxima:", df_categoria["posicion"].max())

print("Posiciones duplicadas:", df_categoria["posicion"].duplicated().sum())

posiciones_esperadas = set(range(1, total_atletas + 1))
posiciones_observadas = set(df_categoria["posicion"].dropna().astype(int))

posiciones_faltantes = sorted(posiciones_esperadas - posiciones_observadas)

print("Número de posiciones faltantes:", len(posiciones_faltantes))
print("Primeras posiciones faltantes:", posiciones_faltantes[:20])

Total esperado: 2350
Total extraído: 2349
Posición mínima: 1
Posición máxima: 2350
Posiciones duplicadas: 0
Número de posiciones faltantes: 1
Primeras posiciones faltantes: [403]


## 9. Exportación de resultados

Guardamos la base de datos de la categoría en formato CSV para reutilizarla en análisis posteriores.

In [ ]:
nombre_archivo = "barcelona2026_hyrox_men.csv"

df_categoria.to_csv(nombre_archivo, index=False)

print("Archivo guardado:", nombre_archivo)

In [80]:
df_pagina_16 = extraer_ranking_pagina(f"{ranking_url}?p=16")

print(len(df_pagina_16))

df_pagina_16[df_pagina_16["posicion"].astype(str) == "1503"]

100


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
2,1503,324,Victor Fablet,None,25-29,1:27:46,None,https://www.hyresult.com/result/LR3MS4JI501362
